# Qwen P2 grid: binary grid-probe balanced accuracy by grid-loudness decile

The grid twin of `wrappers/qwen_analysis/4_loudness_evaluation/probe_accuracy_by_loudness_decile.ipynb`.
Held-out 72 (70 gathered). One row per (trajectory, step, reasoning token): **every** token, so no lens
chose the rows, and the loudness axis covers the real distribution. There are 24 one-vs-rest probes:
3 selection arms (jlens / logitlens / random) x 4 cell classes (empty `_`, wall `#`, agent `A`, goal `G`)
x {lr, mlp}, all at layer 27.

**Input.** `score_probes_per_token.py --probe-type grid_binary`, run as recorded in cell 1 (`SCORED_WITH`).
Each row carries, per probe, the **confusion counts** over that token's 25 cells (`{probe}_tp/_fn/_tn/_fp`).
It also carries both lenses' **grid** loudness at L27 (`{lens}_grid_logmass_L27`, log P(any pruned-vocabulary
grid word)), read from `qwen_p2_heldout_grid_lens`.

**The cells are prepare's.** `grid_binary` draws them with `prepare_activations_for_probing._grid_cell_payload`
using the same seed/pad/cap as `prepared/qwen_p2_grid_heldout`. So these are exactly the rows
`eval_binary_probes_heldout.sh` scored, and **cell 2 must reproduce those 24 JSONs**. If it does not, stop:
nothing below means anything.

**Statistics are the repo's.** `stats.bal_acc_binary_from_counts` pools tp/fn/tn/fp over a bin before dividing:
the mean of pooled recall and pooled specificity. A token's 25 cells therefore carry 25 cells' weight. That
matters here, because on a rare class most tokens have no positive cell at all, and averaging per-token
accuracies collapses to specificity (CLAUDE.md, "the two balanced accuracies"). The 95% bands resample
**trajectory names**, never rows. Deciles come from `stats.qbin` and are re-cut per lens. `columns.axis_label`
names every axis.

**Caveats.** Grids are padded to 15, so padding cells are easy negatives and inflate specificity equally in
every bin. Most grid mass is AXIS (row/column words; `data/jlens/README.md`), so "grid-loud" is largely
"coordinate-loud". Chance is 0.5.

## 1. Load

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from telos_interp.loudness_analysis import columns as cols
from telos_interp.loudness_analysis import stats

# ---- what this run is ---------------------------------------------------------------
TABLE = Path("/workspace/results/qwen_p2_grid/heldout/per_token_scores.csv")
EVAL_JSONS = Path("/workspace/results/qwen_p2_grid/heldout")  # eval_binary_probes_heldout.sh's output
FIG_DIR = TABLE.parent / "figures"
SIGNAL_JSON = Path("/workspace/repo/interp/data/jlens/qwen/grid_tokens_pruned_qwen3-6-35b-a3b.json")
LAYER = 27
SIGNAL = "grid"
LENSES = ("jlens", "logitlens")
N_DECILES = 10
N_BOOT = 300
SEED = 42
EXCLUDE_RADIUS = 2  # verbalisation control: drop grid words and their +-2 neighbours

SCORED_WITH = """
uv run --extra gpu python telos_interp/loudness_analysis/score_probes_per_token.py \\
    --probe-type grid_binary --probe /workspace/probes/qwen_p2_grid/qwen_p2_grid_<arm>_<class>_l27_<lr|mlp>.pt (x24) \\
    --activations-dir /workspace/activations/qwen_p2_heldout \\
    --lens-dir /workspace/activations/qwen_p2_heldout_grid_lens \\
    --trajectories-dir /workspace/trajectories/qwen3.6-35b/replayed_single_step/heldout_72 \\
    --signal-json /workspace/repo/interp/data/jlens/qwen/grid_tokens_pruned_qwen3-6-35b-a3b.json --signal-name grid \\
    --layer 27 --pad-to-size 15 --max-cells 25 --seed 42 \\
    --cache-activations --cache-dir /workspace/results/qwen_p2_local_belief/heldout/_act_cache \\
    --read-threads 16 --device cuda --out /workspace/results/qwen_p2_grid/heldout/per_token_scores.csv
"""

FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

# keep_default_na=False: a decoded token can literally be "NA". Numeric columns are coerced below.
df = pd.read_csv(TABLE, keep_default_na=False, na_values=[""], low_memory=False)
print(f"{len(df):,} token rows, {df['name'].nunique()} trajectories")

# ---- which probes are in the table: discovered from the columns, never hard-coded --------
PROBE_KEYS = sorted(c[: -len("_tp")] for c in df.columns if c.endswith("_tp"))
KEY_RE = re.compile(r"_(?P<arm>jlens|logitlens|random)_(?P<cls>empty|wall|agent|goal)_l\d+_(?P<mt>lr|mlp)$")
META = {k: KEY_RE.search(k).groupdict() for k in PROBE_KEYS}
ARMS = ["jlens", "logitlens", "random"]
CLASSES = ["empty", "wall", "agent", "goal"]
MTYPES = ["lr", "mlp"]
SIDES = ("tp", "fn", "tn", "fp")
COUNT_COLS = [f"{k}_{s}" for k in PROBE_KEYS for s in SIDES]
df[COUNT_COLS] = df[COUNT_COLS].apply(pd.to_numeric).astype(np.int64)
COLOR = dict(zip(ARMS, sns.color_palette("colorblind", len(ARMS)), strict=True))
print(f"{len(PROBE_KEYS)} probes: {sorted({m['arm'] for m in META.values()})} x "
      f"{sorted({m['cls'] for m in META.values()})} x {sorted({m['mt'] for m in META.values()})}")

LOUD = {lens: cols.resolve(df.columns, lens, SIGNAL, LAYER) for lens in LENSES}
for lens, c in LOUD.items():
    df[c] = pd.to_numeric(df[c], errors="coerce")
    print(f"  {lens}: {c}  ({df[c].notna().mean():.1%} of rows have a mass cell)")

def key_for(arm, cls, mt):
    return next(k for k, m in META.items() if (m["arm"], m["cls"], m["mt"]) == (arm, cls, mt))

## 2. Check: the pooled numbers must reproduce the held-out evaluator

Pool every row, then compare with `eval_binary_cognitive_map_probe`'s `global` block for the same probe
on the same manifest. Same cells means the **ground truth must match exactly**: row count, positives
(tp+fn) and negatives (tn+fp). The **verdicts** may differ by a handful of cells out of ~29M. A probability
sitting exactly on the 0.5 threshold can fall either side, because the GPU rounds batches of different shapes
slightly differently (the evaluator and this scorer batch differently). That moves balanced accuracy by ~1e-7.

In [ ]:
check = []
for k in PROBE_KEYS:
    m = META[k]
    ba, rec, spec = stats.bal_acc_binary_from_counts(df, k)
    tp, fn, tn, fp = (int(df[f"{k}_{s}"].sum()) for s in SIDES)
    ref = json.loads((EVAL_JSONS / f"{m['arm']}_{m['cls']}_{m['mt']}.json").read_text())["global"]
    check.append({**m, "bal_acc": ba, "eval_json": ref["balanced_accuracy"], "abs_diff": abs(ba - ref["balanced_accuracy"]),
                  "same_truth": (tp + fn, tn + fp, tp + fn + tn + fp) == (ref["tp"] + ref["fn"], ref["tn"] + ref["fp"], ref["n_rows"]),
                  "cells_flipped": abs(tp - ref["tp"]) + abs(tn - ref["tn"])})
check = pd.DataFrame(check)
print(f"ground truth identical for all {len(check)} probes: {bool(check.same_truth.all())}")
print(f"verdicts: max {check.cells_flipped.max()} cell(s) of {df.n_cells.astype(int).sum():,} differ; max |bal_acc diff| = {check.abs_diff.max():.1e}")
assert check.same_truth.all(), "different ROWS from the evaluator -- stop here"
assert check.cells_flipped.max() <= 10 and check.abs_diff.max() < 1e-5, "verdicts differ beyond threshold jitter -- stop here"
check.pivot_table(index=["cls", "mt"], columns="arm", values="bal_acc").round(4)

## 3. Balanced accuracy by loudness decile, per lens

Counts are summed per (trajectory, decile) first. The bootstrap then resamples trajectory names over
that small frame instead of over 1.17M rows, which gives the same statistic much faster. Decile edges
are fixed at the full-sample edges (as in the direction notebook).

In [ ]:
def with_decile(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    d = frame[np.isfinite(frame[LOUD[lens]])].copy()
    d["decile"] = stats.qbin(d[LOUD[lens]], N_DECILES, labels=False).astype(int) + 1
    return d


def pooled(counts: pd.DataFrame, k: str) -> tuple[float, float, float]:
    return stats.bal_acc_binary_from_counts(counts, k)


def decile_table(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    d = with_decile(frame, lens)
    agg = d.groupby(["decile", "name"])[COUNT_COLS].sum()
    meta = d.groupby("decile").agg(n_tokens=("name", "size"), n_traj=("name", "nunique"), mean_logmass=(LOUD[lens], "mean"))
    rng = np.random.default_rng(SEED)
    names = d["name"].unique()
    draws = [rng.choice(names, size=len(names), replace=True) for _ in range(N_BOOT)]
    rows = []
    for dec, g in agg.groupby(level="decile"):
        g = g.droplevel("decile")
        boot = [g.reindex(pick).fillna(0) for pick in draws]
        for k in PROBE_KEYS:
            ba, rec, spec = pooled(g, k)
            bs = np.array([pooled(b, k)[0] for b in boot])
            rows.append({"lens": lens, "decile": int(dec), **META[k], "probe_key": k, "bal_acc": ba,
                         "lo": np.nanpercentile(bs, 2.5), "hi": np.nanpercentile(bs, 97.5), "recall": rec, "specificity": spec,
                         **meta.loc[dec].to_dict()})
    return pd.DataFrame(rows)


BY_DECILE = pd.concat([decile_table(df, lens) for lens in LENSES], ignore_index=True)
BY_DECILE.to_csv(TABLE.parent / "accuracy_by_grid_loudness_decile.csv", index=False)
print(BY_DECILE[BY_DECILE.lens == "jlens"].drop_duplicates("decile")[["decile", "n_tokens", "n_traj", "mean_logmass"]].to_string(index=False))

### Loudest minus quietest decile, with a trajectory-clustered 95% CI

The same resampled names are used for both ends, so the CI is on the **gap**, not two independent bands.

In [ ]:
def gap_table(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    d = with_decile(frame, lens)
    ends = d[d.decile.isin([1, N_DECILES])].groupby(["decile", "name"])[COUNT_COLS].sum()
    lo_g, hi_g = ends.loc[1], ends.loc[N_DECILES]
    rng = np.random.default_rng(SEED)
    names = d["name"].unique()
    draws = [rng.choice(names, size=len(names), replace=True) for _ in range(N_BOOT)]
    rows = []
    for k in PROBE_KEYS:
        gap = pooled(hi_g, k)[0] - pooled(lo_g, k)[0]
        bs = np.array([pooled(hi_g.reindex(p).fillna(0), k)[0] - pooled(lo_g.reindex(p).fillna(0), k)[0] for p in draws])
        rows.append({"lens": lens, **META[k], "gap_top_minus_bottom": gap,
                     "lo": np.nanpercentile(bs, 2.5), "hi": np.nanpercentile(bs, 97.5)})
    return pd.DataFrame(rows)


GAPS = pd.concat([gap_table(df, lens) for lens in LENSES], ignore_index=True)
GAPS.to_csv(TABLE.parent / "grid_loudness_gap_top_minus_bottom.csv", index=False)
GAPS.pivot_table(index=["lens", "cls", "mt"], columns="arm", values="gap_top_minus_bottom").round(4)

## 4. Figures: one panel per (class, probe family), the three arms overlaid

Read an arm's curve against the others in the same panel: every point in a panel is the same tokens and
the same cells, and only the probe's training selection differs.

In [ ]:
def draw(tab: pd.DataFrame, lens: str, title: str, fname: str) -> None:
    fig, axes = plt.subplots(len(MTYPES), len(CLASSES), figsize=(4.2 * len(CLASSES), 3.6 * len(MTYPES)), sharex=True)
    for r, mt in enumerate(MTYPES):
        for c, cls in enumerate(CLASSES):
            ax = axes[r][c]
            for arm in ARMS:
                t = tab[(tab.lens == lens) & (tab.arm == arm) & (tab.cls == cls) & (tab.mt == mt)].sort_values("decile")
                ax.plot(t.decile, t.bal_acc, marker="o", ms=3.5, lw=1.6, color=COLOR[arm], label=arm)
                ax.fill_between(t.decile, t.lo, t.hi, color=COLOR[arm], alpha=0.15, lw=0)
            ax.axhline(0.5, ls=":", lw=1, color="0.4")
            ax.set_title(f"{cls} / {mt}", fontsize=11)
            if c == 0:
                ax.set_ylabel("balanced accuracy\n(cells pooled)")
            if r == len(MTYPES) - 1:
                ax.set_xlabel(f"{cols.axis_label(lens, SIGNAL, LAYER)} decile\n(1 = quietest)")
            ax.set_xticks(range(1, N_DECILES + 1))
    axes[0][0].legend(title="probe trained on", fontsize=8, title_fontsize=8, loc="best")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{fname}.png", dpi=150)
    print(f"-> {FIG_DIR / f'{fname}.png'}")
    plt.show()


draw(BY_DECILE, "jlens", "Qwen held-out 70, every reasoning token: binary grid probes by J-lens grid loudness", "grid_decile_jlens")

In [ ]:
draw(BY_DECILE, "logitlens", "Qwen held-out 70, every reasoning token: binary grid probes by logit-lens grid loudness", "grid_decile_logitlens")

## 5. Verbalisation control

The lens predicts the *next* tokens, so a token just before ` row` is loud without being a grid word itself.
This drops every token that **is** a pruned-vocabulary grid word, plus its +-`EXCLUDE_RADIUS` neighbours
within the same (trajectory, step), then redraws the J-lens panels. A gradient that survives here is not just
"a grid word is being written".

In [ ]:
vocab = {t for lst in json.loads(SIGNAL_JSON.read_text()).values() for t in lst}
# The table's tokens are raw byte-level BPE ("Ġrow"), the vocabulary decoded text (" row"): normalise first,
# as rollouts/eval_local_belief.py does. Before 2026-09-24 this matched almost nothing.
flag = df["token"].str.replace("Ġ", " ", regex=False).str.replace("Ċ", "\n", regex=False).isin(vocab)
drop = flag.copy()
grouped = flag.groupby([df["name"], df["step"]])
for s in range(1, EXCLUDE_RADIUS + 1):
    drop |= grouped.shift(s, fill_value=False) | grouped.shift(-s, fill_value=False)
quiet = df[~drop]
print(f"grid words: {flag.mean():.2%} of tokens; dropped with radius {EXCLUDE_RADIUS}: {drop.mean():.2%} -> {len(quiet):,} rows")

BY_DECILE_NOVERB = decile_table(quiet, "jlens")
BY_DECILE_NOVERB.to_csv(TABLE.parent / f"accuracy_by_grid_loudness_decile_no_grid_words_r{EXCLUDE_RADIUS}.csv", index=False)
draw(BY_DECILE_NOVERB, "jlens", f"...excluding grid words and their +-{EXCLUDE_RADIUS} neighbours (J-lens)", f"grid_decile_jlens_no_grid_words_r{EXCLUDE_RADIUS}")
gap_table(quiet, "jlens").pivot_table(index=["cls", "mt"], columns="arm", values="gap_top_minus_bottom").round(4)